In [32]:
import requests
import pandas as pd
import time

In [ ]:
def fetch_nasa_power_monthly(lat, lon, region, start=2015, end=2025):
    lat = round(float(lat), 2)
    lon = round(float(lon), 2)

    url = (
        "https://power.larc.nasa.gov/api/temporal/monthly/point?"
        f"parameters=T2M,PRECTOTCORR,RH2M"
        f"&community=AG"
        f"&longitude={lon}"
        f"&latitude={lat}"
        f"&start={start}"
        f"&end={end}"
        f"&format=JSON"
    )

    print(f"\n Requesting NASA POWER for: {region} @ lat={lat}, lon={lon}")
    
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        data = r.json()

        if "properties" not in data:
            print(f"Unexpected response structure for {region}")
            return None
            
        if "parameter" not in data["properties"]:
            print(f"No parameter data for {region}")
            return None

        params = data["properties"]["parameter"]

        def get_series(var_name):
            if var_name not in params:
                return None
            val = params[var_name]
            if not isinstance(val, dict):
                return None
            return val

        t2m = get_series("T2M")
        prec = get_series("PRECTOTCORR")
        rh = get_series("RH2M")

        dates_dict = t2m or prec or rh
        if not dates_dict:
            print(f"No usable data for {region}")
            return None

        dates = sorted(dates_dict.keys())

        df = pd.DataFrame({
            "date": dates,
            "temperature": [t2m.get(d) if t2m else None for d in dates],
            "precipitation": [prec.get(d) if prec else None for d in dates],
            "humidity": [rh.get(d) if rh else None for d in dates],
            "region": region,
            "lat": lat,
            "lon": lon
        })

        # Parse dates
        df["date"] = pd.to_datetime(df["date"], format="%Y%m", errors="coerce")
        
        df = df.dropna(subset=["date"])
        
        print(f"Successfully fetched {len(df)} records for {region}")
        return df
    
    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error for {region}: {e}")
        return None
    except requests.exceptions.Timeout:
        print(f"Timeout for {region}")
        return None
    except Exception as e:
        print(f"Unexpected error for {region}: {e}")
        return None

In [35]:
mp = pd.read_csv("Data/Market_Prices.csv")

regions = mp.groupby("mkt_name")[["lat", "lon"]].first().reset_index()
print(f"\nFound {len(regions)} unique markets to query")

all_regions = []
failed_regions = []

for idx, row in regions.iterrows():
    region = row["mkt_name"]
    lat = row["lat"]
    lon = row["lon"]
    
    if idx > 0:
        time.sleep(1)
    
    print(f"\nProgress: {idx + 1}/{len(regions)}")
    
    reg_df = fetch_nasa_power_monthly(lat, lon, region)
    
    if reg_df is not None:
        all_regions.append(reg_df)
    else:
        failed_regions.append(region)


Found 45 unique markets to query

Progress: 1/45

 Requesting NASA POWER for: Afgooye @ lat=2.14, lon=45.12
Successfully fetched 132 records for Afgooye

Progress: 2/45

 Requesting NASA POWER for: Afmadow @ lat=0.51, lon=42.07
Successfully fetched 132 records for Afmadow

Progress: 3/45

 Requesting NASA POWER for: Baidoa @ lat=3.12, lon=43.65
Successfully fetched 132 records for Baidoa

Progress: 4/45

 Requesting NASA POWER for: Bakaara @ lat=2.05, lon=45.32
Successfully fetched 132 records for Bakaara

Progress: 5/45

 Requesting NASA POWER for: Balcad @ lat=2.36, lon=45.39
Successfully fetched 132 records for Balcad

Progress: 6/45

 Requesting NASA POWER for: Belet Xaawo @ lat=3.78, lon=41.89
Successfully fetched 132 records for Belet Xaawo

Progress: 7/45

 Requesting NASA POWER for: Beletweyne @ lat=4.74, lon=45.2
Successfully fetched 132 records for Beletweyne

Progress: 8/45

 Requesting NASA POWER for: Berbera @ lat=10.44, lon=45.01
Successfully fetched 132 records for Berb

In [ ]:
if all_regions:
    final_df = pd.concat(all_regions, ignore_index=True)
    
    output_file = "Climate_Data.csv"
    final_df.to_csv(output_file, index=False)
    print("\nPreview of data:")
    print(final_df.head(10))
    print("\nData summary:")
    print(final_df.describe())
    
    if failed_regions:
        print(f"\n Failed regions: {', '.join(failed_regions[:10])}")
        if len(failed_regions) > 10:
            print(f"   ... and {len(failed_regions) - 10} more")
else:
    print("\n No regions returned data! Check your parameters and coordinates.")


Preview of data:
        date  temperature  precipitation  humidity   region   lat    lon
0 2015-01-01        27.22           0.00     64.38  Afgooye  2.14  45.12
1 2015-02-01        27.77           0.00     65.96  Afgooye  2.14  45.12
2 2015-03-01        28.34           0.75     67.10  Afgooye  2.14  45.12
3 2015-04-01        28.66           2.56     74.50  Afgooye  2.14  45.12
4 2015-05-01        27.92           2.72     78.24  Afgooye  2.14  45.12
5 2015-06-01        27.22           0.68     74.23  Afgooye  2.14  45.12
6 2015-07-01        26.19           1.40     73.45  Afgooye  2.14  45.12
7 2015-08-01        26.61           0.89     71.72  Afgooye  2.14  45.12
8 2015-09-01        27.55           0.15     70.51  Afgooye  2.14  45.12
9 2015-10-01        28.14           2.25     74.46  Afgooye  2.14  45.12

Data summary:
                                date  temperature  precipitation     humidity  \
count                           5808  5808.000000    5808.000000  5808.000000   
me

: 